In [ ]:
import pandas as pd
import pandas_ta as ta
import pyotp
from datetime import datetime, timedelta
import time
import csv
import asyncio
import nest_asyncio
import logging
import websocket as ws_client
import xarray as xr

import threading

from NorenRestApiPy.NorenApi import NorenApi

class ShoonyaApiPy(NorenApi):
    def __init__(self):
        NorenApi.__init__(self, host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')        
        global api
        api = self

import logging
#logging.basicConfig(level=logging.DEBUG)
logging.getLogger('websocket').setLevel(logging.INFO)

usercred = pd.read_excel("usercred.xlsx")
user    = usercred.iloc[0,0]
pwd     = usercred.iloc[0,1]
vc      = usercred.iloc[0,2]
app_key = usercred.iloc[0,3]
imei    = usercred.iloc[0,4]
qr = usercred.iloc[0,5]
factor2 = pyotp.TOTP(qr).now()

api = ShoonyaApiPy()
ret = api.login(userid=user, password=pwd, twoFA=factor2, vendor_code=vc, api_secret=app_key, imei=imei)

In [ ]:
def append_to_csv(trade_log):
    with open('trading_log.csv', 'a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(trade_log)

In [ ]:
def calculate_indicators(token, df):
    try:
        # Calculate the indicators
        ema_short = ta.ema(df['close'], length=3)
        ema_long = ta.ema(df['close'], length=10)
        # ema_long_x = ta.ema(df['close'], length=20)
        alma_short = ta.alma(df['close'], length=20, sigma=6, distribution_offset=0.85)

        # Initialize the data for the token if it does not exist
        if token not in indicator_data:
            indicator_data[token] = {
                'EMA_short': pd.Series(dtype='float64'),
                'EMA_long': pd.Series(dtype='float64'),
                'ALMA_short': pd.Series(dtype='float64'),
            }

        # Append new indicator data to existing data
        if ema_short is not None:
            indicator_data[token]['EMA_short'] = pd.concat([indicator_data[token]['EMA_short'], ema_short])
        if ema_long is not None:
            indicator_data[token]['EMA_long'] = pd.concat([indicator_data[token]['EMA_long'], ema_long])
        # if ema_long_x is not None:
        #     indicator_data[token]['EMA_long_x'] = pd.concat([indicator_data[token]['EMA_long_x'], ema_long])        
        if alma_short is not None:
            indicator_data[token]['ALMA_short'] = pd.concat([indicator_data[token]['ALMA_short'], alma_short])

    except Exception as e:
        print(f"Error calculating indicators for token {token}: {e}")


In [ ]:
current_position = 'none'
async def exit_logic(token):
    # Implement your exit logic
    #print("i m ex")
    pass

async def trailing_stop_loss(token):
    # Implement your trailing stop loss logic
    #print("i m sl")
    pass

async def profit_booking(token):
    # Implement your profit booking logic
    #print("i m pb")
    pass

async def entry_logic(token, latest_timestamp, resampled_df, indicator_dict):
    global current_position  # Assuming you have a global variable to track the current position

    # Extract the necessary indicators for the latest timestamp
    ema_short = indicator_dict['EMA_short']
    ema_long = indicator_dict['EMA_long']
    alma_short = indicator_dict['ALMA_short']

    if len(ema_short) < 6 or len(ema_long) < 20 or len(alma_short) < 25:
        return  # Not enough data to make a decision

    # Get the values for the current and previous timestamp
    current_ema_short = ema_short.loc[latest_timestamp]
    current_ema_long = ema_long.loc[latest_timestamp]
    current_alma_short = alma_short.loc[latest_timestamp]

    previous_timestamp = resampled_df.index[-2]
    previous_ema_short = ema_short.loc[previous_timestamp]
    previous_ema_long = ema_long.loc[previous_timestamp]

    # Check for a long entry
    if (current_ema_short > current_ema_long) and (previous_ema_short <= previous_ema_long):
        if current_ema_short > current_alma_short and current_ema_long > current_alma_short:
            if current_position != 'long':
                print(f"Long entry for {token} at {latest_timestamp}")
                current_position = 'long'
                append_to_csv(["Enter Long", token, latest_timestamp])
                # Place your long entry order here
                # ...

    # Check for a short entry
    elif (current_ema_short < current_ema_long) and (previous_ema_short >= previous_ema_long):
        if current_ema_short < current_alma_short and current_ema_long < current_alma_short:
            if current_position != 'short':
                print(f"Short entry for {token} at {latest_timestamp}")
                current_position = 'short'
                append_to_csv(["Enter Short", token, latest_timestamp])
                # Place your short entry order here
                # ...


In [ ]:
async def trading_logic():
    global indicator_data, trading_active, resampled_data, last_processed_timestamp

    while True:
        if trading_active:
            for token in resampled_data.keys():
                if token in indicator_data:
                    resampled_df = resampled_data[token]
                    indicator_dict = indicator_data[token]

                    # Get the latest timestamp in resampled_data
                    latest_timestamp = resampled_df.index[-1]

                    # Initialize last_processed_timestamp for this token if not present
                    if token not in last_processed_timestamp:
                        last_processed_timestamp[token] = None

                    # Debug: Check indicator data for None values
                    for ind in ['EMA_short', 'EMA_long', 'ALMA_short']:
                        if indicator_dict.get(ind) is None:
                            print(f"Indicator {ind} for token {token} is None at {latest_timestamp}")

                    # Ensure the latest timestamp exists in all required indicators
                    if all(indicator_dict.get(ind) is not None and latest_timestamp in indicator_dict[ind].index for ind in ['EMA_short', 'EMA_long', 'ALMA_short']):
                        # Ensure it is a new resample period and not already processed
                        if latest_timestamp != last_processed_timestamp[token]:
                            # Check if the timestamp corresponds to the end of a resample period
                            if latest_timestamp.second % 15 == 0:  # Assuming 15-second resample period
                                print(f"Trading logic working for {token} at {latest_timestamp}")

                                # Implement your entry logic based on the end of the resample period
                                asyncio.create_task(entry_logic(token, latest_timestamp, resampled_df, indicator_dict))

                                # Update last processed timestamp for this token
                                last_processed_timestamp[token] = latest_timestamp

                    # Run exit, trailing SL, and profit booking logic asynchronously
                    asyncio.create_task(exit_logic(token))
                    asyncio.create_task(trailing_stop_loss(token))
                    asyncio.create_task(profit_booking(token))
                else:
                    print(f"Indicator data not available for token {token}")
        else:
            print("Trading logic paused")
        
        await asyncio.sleep(1)


In [ ]:
import asyncio
import threading
from datetime import datetime
import pandas as pd
import nest_asyncio

trading_active = False
feed_opened = False
feedJson = {}
resample_frequency = "15s"
resampling_lock = threading.Lock()
last_resample_time = {}
resampled_data = {}
indicator_data = {}  # New dictionary to store indicator values
last_processed_timestamp = {}

if 'resampled_data' not in globals():
    resampled_data = {}
if 'last_resample_time' not in globals():
    last_resample_time = {}

def event_handler_feed_update(tick_data):
    print(tick_data)
    try:
        if 'lp' in tick_data and 'tk' in tick_data:
            timest = datetime.fromtimestamp(int(tick_data['ft'])).isoformat()
            token = tick_data['tk']

            with resampling_lock:
                if token not in feedJson:
                    feedJson[token] = []
                    last_resample_time[token] = timest

                feedJson[token].append({'ltp': float(tick_data['lp']), 'tt': timest})
    except (KeyError, ValueError) as e:
        print(f"Error processing tick data: {e}")

async def resample_ticks():
    
    while True:
        if not feedJson:
            await asyncio.sleep(1)
            continue

        with resampling_lock:
            for token, ticks in feedJson.items():
                try:
                    if ticks:
                        # Create a DataFrame with the new ticks
                        df_new = pd.DataFrame(ticks)
                        df_new["tt"] = pd.to_datetime(df_new["tt"])
                        df_new.set_index("tt", inplace=True)

                        # Determine the current resample interval
                        current_resample_interval = df_new.index.max().floor(resample_frequency)

                        # Initialize or update the resampled DataFrame
                        if token not in resampled_data:
                            # Initialize the DataFrame with the first resample
                            df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()
                            resampled_data[token] = df_resampled
                            last_resample_time[token] = df_resampled.index.max()
                        else:
                            # Resample the new ticks
                            df_resampled = df_new['ltp'].resample(resample_frequency).ohlc()

                            # Update the existing resampled DataFrame with new data
                            for idx, row in df_resampled.iterrows():
                                if idx in resampled_data[token].index:
                                    resampled_data[token].loc[idx, 'high'] = max(resampled_data[token].loc[idx, 'high'], row['high'])
                                    resampled_data[token].loc[idx, 'low'] = min(resampled_data[token].loc[idx, 'low'], row['low'])
                                    resampled_data[token].loc[idx, 'close'] = row['close']
                                else:
                                    resampled_data[token].loc[idx] = row

                            # Update the last resampled time
                            last_resample_time[token] = df_resampled.index.max()

                        #print(f"Resampled data for token {token}:\n{resampled_data[token]}")
                        # Calculate indicators for the updated resampled data
                        calculate_indicators(token, resampled_data[token])                       

                    feedJson[token] = []

                except Exception as e:
                    if isinstance(e, KeyError) and e.args[0] == 'tt':
                        print(f"Market likely closed for token {token}")
                    else:
                        print(f"Error resampling data for token {token}: {e}")

        await asyncio.sleep(1)
        

def event_handler_order_update(tick_data):
    print(f"Order update {tick_data}")

def open_callback():
    global feed_opened
    feed_opened = True

# Assuming `api` is defined and starts the WebSocket connection
api.start_websocket(
    order_update_callback=event_handler_order_update,
    subscribe_callback=event_handler_feed_update,
    socket_open_callback=open_callback
)
while not feed_opened:
    pass
api.subscribe(['MCX|428009'])

async def main():
    resample_task = asyncio.create_task(resample_ticks())
    trading_task = asyncio.create_task(trading_logic())

    await asyncio.gather(resample_task, trading_task)

def set_trading_active(value):
    global trading_active
    trading_active = value
    #print(f"Trading active set to {trading_active}")

# Get the current event loop
loop = asyncio.get_event_loop()

if loop.is_running():
    nest_asyncio.apply()
    asyncio.create_task(main())
else:
    loop.run_until_complete(main())

# Example: Dynamically change the trading_active flag
set_trading_active(True)
asyncio.run(asyncio.sleep(5))  # Simulate some delay

In [ ]:

set_trading_active(False)
asyncio.run(asyncio.sleep(5))  # Simulate some delay

In [ ]:
import csv
# Open the CSV file
with open("data.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)

    # Write the header row
    writer.writerow(indicator_data.keys())

    # Write the values row
    writer.writerow(indicator_data.values())

In [ ]:
resampled_data

In [ ]:
feedJson

In [ ]:
def check_long_entry_conditions(df, token):
    ema1_latest = df['ema1'].iloc[-1]
    ema2_latest = df['ema2'].iloc[-1]
    ema3_latest = df['ema3'].iloc[-1]
    ema1_previous = df['ema1'].iloc[-2]
    ema2_previous = df['ema2'].iloc[-2]
    
    if len(df) > ema_length and ema1_latest > ema2_latest and ema1_previous <= ema2_previous and ema1_latest >= ema3_latest and ema2_latest >= ema3_latest:
        if current_positions.get(token) is None or current_positions[token]['position'] != 'long':
            current_positions[token] = {
                'position': 'long',
                'entry_price': df['close'].iloc[-1],
                'entry_time': df.index[-1],
                'initial_sl': df['close'].iloc[-1] - initial_sl_points,
                'sl_moved': False
            }
            print(f"Enter Long: {token} at {current_positions[token]['entry_time']}, Price: {current_positions[token]['entry_price']}")
            append_to_csv(["Enter Long", token, current_positions[token]['entry_time'], current_positions[token]['entry_price']])

def check_long_exit_conditions(df, token, current_price):
    entry_price = current_positions[token]['entry_price']
    initial_sl = current_positions[token]['initial_sl']
    sl_moved = current_positions[token]['sl_moved']
    
    if not sl_moved and current_price >= entry_price + move_sl_points:
        current_positions[token]['initial_sl'] = entry_price
        current_positions[token]['sl_moved'] = True
        print(f"SL moved to entry price for long position on {token}")

    if current_price <= initial_sl or current_price >= entry_price + profit_booking_points or (df['ema1'].iloc[-1] < df['ema2'].iloc[-1] and df['ema1'].iloc[-2] > df['ema2'].iloc[-2]):
        exit_timestamp = df.index[-1]
        exit_price = current_price
        print(f"Exit Long: {token} at {exit_timestamp}, Price: {exit_price}")
        append_to_csv(["Exit Long", token, exit_timestamp, exit_price])
        current_positions[token] = None

def check_short_entry_conditions(df, token):
    ema1_latest = df['ema1'].iloc[-1]
    ema2_latest = df['ema2'].iloc[-1]
    ema3_latest = df['ema3'].iloc[-1]
    ema1_previous = df['ema1'].iloc[-2]
    ema2_previous = df['ema2'].iloc[-2]
    
    if len(df) > ema_length and ema1_latest < ema2_latest and ema1_previous >= ema2_previous and ema1_latest <= ema3_latest and ema2_latest <= ema3_latest:
        if current_positions.get(token) is None or current_positions[token]['position'] != 'short':
            current_positions[token] = {
                'position': 'short',
                'entry_price': df['close'].iloc[-1],
                'entry_time': df.index[-1],
                'initial_sl': df['close'].iloc[-1] + initial_sl_points,
                'sl_moved': False
            }
            print(f"Enter Short: {token} at {current_positions[token]['entry_time']}, Price: {current_positions[token]['entry_price']}")
            append_to_csv(["Enter Short", token, current_positions[token]['entry_time'], current_positions[token]['entry_price']])

def check_short_exit_conditions(df, token, current_price):
    entry_price = current_positions[token]['entry_price']
    initial_sl = current_positions[token]['initial_sl']
    sl_moved = current_positions[token]['sl_moved']
    
    if not sl_moved and current_price <= entry_price - move_sl_points:
        current_positions[token]['initial_sl'] = entry_price
        current_positions[token]['sl_moved'] = True
        print(f"SL moved to entry price for short position on {token}")

    if current_price >= initial_sl or current_price <= entry_price - profit_booking_points or (df['ema1'].iloc[-1] > df['ema2'].iloc[-1] and df['ema1'].iloc[-2] < df['ema2'].iloc[-2]):
        exit_timestamp = df.index[-1]
        exit_price = current_price
        print(f"Exit Short: {token} at {exit_timestamp}, Price: {exit_price}")
        append_to_csv(["Exit Short", token, exit_timestamp, exit_price])
        current_positions[token] = None

def process_token_data(token, df):
    current_feed_json = {**feedJson}
    if len(df) > ema_length and 'ema1' in df.columns and 'ema2' in df.columns and 'ema3' in df.columns:
        
        check_long_entry_conditions(df, token)
        
        if current_positions.get(token) and current_positions[token]['position'] == 'long':
            current_price = current_feed_json.get(token, {}).get('ltp')
            check_long_exit_conditions(df, token, current_price)
        
        check_short_entry_conditions(df, token)
        
        if current_positions.get(token) and current_positions[token]['position'] == 'short':
            current_price = current_feed_json.get(token, {}).get('ltp')
            check_short_exit_conditions(df, token, current_price)


In [ ]:
import csv

# Assuming last_resampled_data is defined and populated with data

# Specify the CSV file path
csv_file = 'last_resampled_data1.csv'

try:
    # Open the CSV file in write mode
    with open(csv_file, 'w', newline='') as file:
        writer = csv.writer(file)

        # Write headers for each token's data
        writer.writerow(['Token', 'Timestamp', 'Open', 'High', 'Low', 'Close', 'EMA1', 'EMA2' , 'EMA3'])

        # Iterate through each token and its corresponding DataFrame
        for token, df in last_resampled_data.items():
            # Extract relevant columns for writing to CSV
            for index, row in df.iterrows():
                writer.writerow([token, index, row['open'], row['high'], row['low'], row['close'], row['ema1'], row['ema2'], row['ema3']])

    print(f"Data successfully written to {csv_file}")

except Exception as e:
    print(f"Error writing data to CSV: {e}")


In [ ]:
# def calculate_indicators(token, df):
#     try:
#         # Calculate the indicators
#         ema_short = ta.ema(df['close'], length=3)
#         ema_long = ta.ema(df['close'], length=10)
#         alma_short = ta.alma(df['close'], length=20, sigma=6, distribution_offset=0.85)

#         # Initialize the data for the token if it does not exist
#         if token not in indicator_data:
#             indicator_data[token] = {
#                 'EMA_short': pd.Series(dtype='float64'),
#                 'EMA_long': pd.Series(dtype='float64'),
#                 'ALMA_short': pd.Series(dtype='float64'),
#             }

#         # Ensure that new calculations are not None
#         if ema_short is not None:
#             ema_short_new = ema_short.loc[~ema_short.index.isin(indicator_data[token]['EMA_short'].index)]
#         else:
#             ema_short_new = pd.Series(dtype='float64')

#         if ema_long is not None:
#             ema_long_new = ema_long.loc[~ema_long.index.isin(indicator_data[token]['EMA_long'].index)]
#         else:
#             ema_long_new = pd.Series(dtype='float64')

#         if alma_short is not None:
#             alma_short_new = alma_short.loc[~alma_short.index.isin(indicator_data[token]['ALMA_short'].index)]
#         else:
#             alma_short_new = pd.Series(dtype='float64')

#         # Concatenate new indicator data to existing data
#         if not ema_short_new.empty:
#             indicator_data[token]['EMA_short'] = pd.concat([indicator_data[token]['EMA_short'], ema_short_new])
#         if not ema_long_new.empty:
#             indicator_data[token]['EMA_long'] = pd.concat([indicator_data[token]['EMA_long'], ema_long_new])
#         if not alma_short_new.empty:
#             indicator_data[token]['ALMA_short'] = pd.concat([indicator_data[token]['ALMA_short'], alma_short_new])

#     except Exception as e:
#         print(f"Error calculating indicators for token {token}: {e}")

In [ ]:
from deephaven_server import Server
server = Server()
server.start()

# Here is a simple example to create a new (growing) time table
from deephaven import time_table
table_t = time_table('00:00:02').update(formulas = ["Row = 1", "Squared = Row^2"])